# Can teacher training copy a fine-tune with no trigger?

*E37, `experiments/teacher_spread`, 2026-10-26. Record: `reports/analyses/2026-10-26_analysis_teacher-spread.md`.
Follows E34, `reports/analyses/2026-10-23_analysis_teacher-training.md`, and uses E36's lowercase fine-tune. One run,
provisional. Independently checked 2026-10-26, `reports/reviews/2026-10-26_review_teacher-spread.md`; corrections
applied.*

> **Something unexpected happened that I cannot explain.** We predicted the lowercase copy would pass the adversarial
> check, as E34's copies of trigger fine-tunes had. It read 2.20 times the unchanged model, above the fail line of
> 1.61. A second filed prediction, that the adjoint split would lose most of the behaviour, also failed. That one
> drew on a different measure in E34, so it is a smaller surprise than it looks.

## The answer

Not as filed. Teacher training copied the lowercase fine-tune almost exactly: the copy keeps all of its held-out gain
and does the same damage elsewhere. But the standard mask adversary hurts it twice as much as the unchanged model,
past the fail line, at both batch orders and from ordinary text. The copy was still learning when read. The adjoint
split kept most of the gain and passed, but it barely changes predictions on ordinary text, which may be why it passes.

## What we were trying to test

**The model.** The released 67M Pile model with its VPD decomposition. Each matrix is $W = UV^\top + \Delta$; the
columns of $U$ and $V$ are the alive subcomponents' write and read vectors. Γ and Δ are frozen throughout.

**The teacher.** E36's lowercase fine-tune. All 24 decomposed matrices were trained on the next-token loss at every
position of lowercased Pile text, where capitals never come. Adam at rate 3e-5, 300 steps. Its update is
$\delta W$. Every fine-tune that teacher training copied before acted at one trigger token; this one changes
predictions everywhere, as post-training of a larger model would.

**Teacher training** trains the alive $U$ and $V$ to minimise $\mathrm{KL}(\text{teacher} \,\|\, \text{student})$ on
unlabelled text, Γ and Δ frozen, for 1,000 steps with E34's recipe.

**Six states**, each with the alternative it rules out:

- the **unchanged model**, the floor every ratio is taken against;
- **do-nothing**: $\delta W$ goes into Δ, the subcomponents stay as released. The reference: any map should do at
  least this well;
- the **adjoint split** of $\delta W$, the cheap closed-form map, which lost the trigger behaviours in E34;
- the **lowercase copy**, trained on the lowercase rows the teacher trained on;
- the **plain-text copy**, the same documents with capitals kept: is ordinary text enough to copy a style?
- the **lowercase copy at a second batch order**: the copy's noise, and whether two copies land on the same weights.

**What is read**, one symbol per line:

- **behaviour kept** $s$: the state's gain in mean log-probability on held-out lowercase text, over the teacher's.
  0 is the unchanged model, 1 the teacher;
- **teacher gap share**: $\mathrm{KL}(\text{teacher} \,\|\, \text{state})$ over the unchanged model's. 0 is the teacher;
- **damage**: $\mathrm{KL}(W_0 \,\|\, \text{state})$ on plain Pile text, how much the state changes ordinary text;
- **adversarial ratio**: an adversary searches 20 steps for the masks that pull the masked model furthest from the
  target. The ratio is its KL over the unchanged model's; 1 is no more fragile;
- **cosine** between two weight changes, over all 24 matrices at once.

The adversarial check passes at or under 1.40 and fails at or over 1.61 on this card; between is inconclusive. The
adversary may also switch off Δ. In do-nothing Δ holds the whole update, so do-nothing's ratio is high by construction.

## What we expected, and why

- **The lowercase copy keeps at least 0.9 of the gain** and leaves at most a tenth of the gap. Every E34 copy
  reached its teacher.
- **It passes the adversarial check.** All of E34's Adam copies did, even where their teachers did not.
- **Its damage on plain text is half to twice the teacher's**: a faithful copy shares the side effects.
- **The adjoint split keeps under half the gain.** On five of E34's six trigger teachers it kept a quarter or less of
  the target probability, a probability rather than a share of log-probability gain.
- **The plain-text copy also keeps the behaviour**, at low confidence.
- **Two batch orders land on the same weight change**, a cosine of at least 0.8; under 0.5 would mean one of many.

## What we observed

### Does the copy keep the behaviour, and is it faithful?

![behaviour and faithfulness](../experiments/teacher_spread/figures/behaviour_faithful.png)

**Reading it.** It keeps the behaviour and fails the adversarial check. Both lowercase copies, in blue, sit at the
right edge with the teacher, keeping 0.998 and 0.994 of the gain. Both sit above the red fail line, at 2.20 and 1.88.
The plain-text copy, orange, keeps 0.92 and also fails, at 1.74. The adjoint split, grey, keeps 0.80 of the gain and
sits at 1.05, inside the unchanged model's own spread. Do-nothing, black, reads 4.87, high by construction.

The copy is close to the teacher in every other way. It leaves 0.04 of the gap and does the teacher's damage on plain
Pile text. The adjoint split leaves 0.41 of the gap and a fifth of the damage. So the adjoint split raises the
log-probability of lowercase text without matching the teacher's predictions, while the copy matches them.

**An alternative reading, untested.** The ratio rises with damage on ordinary text. The adjoint split changes Pile
predictions by 0.05 nats and passes; the copies change them by 0.23 nats, as the teacher does, and fail. This
teacher changes ordinary text several times more than E34's did. Relative to its own teacher the copy reads 0.39 to
0.45 here, against about 0.6 to 1.2 for E34's copies. By that measure this copy is no worse than E34's.

Judged against do-nothing, as the anchor card's option b would, every copy passes, but that flatters every map.

The copies were still learning: their training KL fell by a third between steps 500 and 1,000. The filed decision tree,
read literally, sends this outcome to "under-trained, rerun longer". The 2.20 is a reading of an unfinished copy at
one step.

| state | behaviour kept | gap share | damage, nats | adversarial |
|---|---|---|---|---|
| do-nothing | 1 | 0 | 0.235 | 4.87 |
| adjoint split | 0.80 | 0.41 | 0.050 | 1.05 |
| lowercase copy | 0.998 | 0.04 | 0.234 | 2.20 |
| lowercase copy, 2nd order | 0.994 | 0.04 | 0.234 | 1.88 |
| plain-text copy | 0.92 | 0.08 | 0.224 | 1.74 |

### Does the failure sit in one half of the rows?

![blocks](../experiments/teacher_spread/figures/blocks.png)

**Reading it.** No: both halves put the lowercase copies over the fail line. The harness averages its adversary over
two blocks of 128 rows. The lowercase copy reads 1.69 on the first block and 2.68 on the second. The plain-text copy
reads 1.60 on the first, just under the fail line, and 1.86 on the second. The hollow diamonds are the budget reading,
the same search on the first block's rows, read in full precision. It agrees with the first block within about a
tenth for every state. So it is not a second adversary: this run has one adversary at one seed per state.

### Do two copies land on the same weight change?

![placement](../experiments/teacher_spread/figures/placement.png)

**Reading it.** Yes. The two batch orders' weight changes have a cosine of 0.93, above the 0.8 line, and the lowest
matrix reads 0.91. The lowercase and plain-text copies read 0.81 against each other. Each copy points along the
fine-tune at 0.44, level with the adjoint split. E34's copies read 0.08 to 0.29 against their fine-tunes. A random
direction reads zero. Between batch orders the behaviour kept moved by 0.004. The adversarial ratio moved by 14%,
inside the spread E34 saw.

## What we learned

Teacher training can copy a broad fine-tune: the copy matches the teacher's predictions to a twentieth of the
original gap and shares its side effects. But at step 1,000 the copy is not faithful by the fixed line, at two batch
orders and from two kinds of text, under one adversary. Relative to its teacher it is no worse than E34's copies were;
this teacher simply changes ordinary text far more, and the ratio rises with that change. Whether that explains the
failure is untested. The adjoint split keeps most of a broad gain and passes, while barely changing ordinary text.

## How compelling is the evidence

- **One teacher, one fine-tune seed, one synthetic style**, read at one step on copies still learning.
- **One adversary, one seed per state.** The budget reading repeats the first block. What was varied is batch order
  and kind of text, and all three copies fail.
- **Seed noise.** A copy's batch-order spread in the ratio is 14% here and up to 25% in E34. The lowercase copy sits
  37% above the fail line; the plain-text copy only 8% above it.
- **Do-nothing's reading is high by construction**, so every comparison with it flatters the map.
- **Behaviour kept is a mean over positions.** The adjoint split shows the mean can be kept with different
  predictions.
- **The damage reading rests on three damage levels**; it is a reading, not a test.

## Surprises

- **The copy fails the adversarial check, against a filed prediction that it would pass.** All of E34's Adam copies
  had read under 1.37. We have no tested cause. The copy was still learning, the update is broad, and the teacher
  changes ordinary text several times more than E34's did.
- **The adjoint split keeps 0.80 of the gain, against a filed prediction of under half.** The verdict stands by the
  filed rule. But the expectation came from E34's target probabilities, another measure. Converted to shares of
  log-probability gain, the independent check estimates E34's adjoint split kept about 0.5 to 0.6.
- **The copies point along the fine-tune more than E34's did**, 0.44 against an expected 0.08 to 0.3.
- **The copies are three quarters of the teacher's size**, not the expected half.

## What next

The options on card `decision-broad-copy`, in its order:

1. **Train the copy three times longer: the filed tree's own branch, and the cheapest step.** About $2, reading the
   adversarial check at several steps and two adversary seeds. A ratio falling under the fail line as the loss
   flattens means under-training; one that stays means the failure is the copy's.
2. **Judge the copy against do-nothing**, option b of the anchor card. Every copy here then passes, but do-nothing's
   reading is high by construction.
3. **Take the adjoint split as the map for broad updates, or start the copy from it.** A copy started there and read
   at several steps would show whether matching the teacher's changes on ordinary text brings the fragility.
4. **If the first step leaves the failure standing, test whether the frozen Γ or the subcomponents' span limits the
   copy**, E15's refit at about $9.

## Scorecard

| id | claim | read | verdict |
|---|---|---|---|
| P1 | lowercase copy keeps the behaviour, s ≥ 0.9 | 0.998 | holds |
| P2 | lowercase copy close to the teacher, gap share ≤ 0.10 | 0.044 | holds |
| P3 | lowercase copy passes the harness, ≤ 1.40 | 2.20, fail line 1.61 | **fails, reversal** |
| P4 | damage 0.5 to 2 times the teacher's | 0.996 | holds |
| P5 | adjoint split loses the behaviour, s < 0.5 | 0.804; expectation from another measure | **fails** |
| P6 | plain-text copy keeps the behaviour, s ≥ 0.9 | 0.919 | holds |
| P7 | one placement, cosine ≥ 0.8 | 0.929 | holds |

All three controls passed. The unchanged model read its known floor, 0.0748. The teacher reproduced E36's gain within
0.00015, and do-nothing read exactly as the teacher.

## Not run

- A second fine-tune seed or a second broad teacher.
- Copies read at other steps, or trained longer.
- The plain-text copy at a second batch order; the second batch order's budget reading.
- A second adversary seed on any updated state; an independent second adversary; adversaries past 20 steps.
- A test of the damage reading: maps with matched damage on ordinary text.
- A copy started from the adjoint split; E15's Γ refit.
- LoRA teachers on this task; the larger model.

**Provenance.** RunPod H100 80GB HBM3, about 21 min and $1.40, job 20:22 to 20:36, exit 0. Results
`experiments/teacher_spread/results/teacher_spread.json`; predictions `results/predictions.json`, filed 19:53:47,
unedited; figures from `experiments/teacher_spread/plot.py`. Full tables, block readings and hashes are in the report.